In [1]:
import pandas as pd
import numpy as np
import os
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

## Baseline Model (Method 1)

#### Economic Data Preprocess

In [2]:
def process_economic_data(base_path):
    files = {
        'HOUST':    ('HOUST.csv',    'HOUST',     12),
        'HPI':      ('HPI.csv',      'CSUSHPISA', 12),
        'CPI':      ('CPIAUCSL.csv', 'CPIAUCSL',  12),
        'PCE':      ('PCE.csv',      'PCE',       12),
        'FEDFUNDS': ('FEDFUNDS.csv', 'FEDFUNDS',  1),
        'SPREAD':   ('T10Y3MM.csv',  'T10Y3MM',   1),
        'UMICH':    ('UMICH.csv',    'MICH',      1),
        'NFP':      ('NFP.csv',      'PAYEMS',    1),
        'BALANCE':  ('BOPGSTB.csv',  'BOPGSTB',   1),
        'UNRATE':   ('UNRATE.csv',   'UNRATE',    1),
        'RSALES':   ('RSAFS.csv',    'RSAFS',     12)
    }

    series_list = []
    print(">> Processing Monthly Economic Data...")
    
    for key, (filename, col_name, lag) in files.items():
        file_path = os.path.join(base_path, filename)
        if os.path.exists(file_path):
            # Load and set Index
            df = pd.read_csv(file_path, parse_dates=['observation_date'])
            df = df.set_index('observation_date').sort_index()
            
            # Handle potential column name mismatches
            if col_name not in df.columns: col_name = df.columns[0]
            series = df[col_name]

            # Transformation Logic
            if key == 'RSALES':
                transformed = series.diff(12)
                transformed.name = 'Rsales_diff_year'
            elif key in ['CPI', 'PCE']:
                transformed = series.pct_change(lag) * 100
                transformed.name = f'{key}_Inflation_Rate'
            else:
                suffix = "year" if lag == 12 else "prev"
                transformed = series.diff(lag)
                transformed.name = f'{key}_diff_{suffix}'
            
            series_list.append(transformed)

            # Special Feature: Previous Decision (Proxy)
            if key == 'FEDFUNDS':
                rate_change = series.diff(1)
                decision_proxy = pd.cut(rate_change, 
                                      bins=[-np.inf, -0.125, 0.125, np.inf], 
                                      labels=[-1, 0, 1]).astype(float)
                prev_decision = decision_proxy.shift(1)
                prev_decision.name = 'prev_decision'
                series_list.append(prev_decision)

    # Combine and Drop NaNs
    if not series_list: return pd.DataFrame()
    df_econ = pd.concat(series_list, axis=1).dropna()
    
    # Standardize (Z-Score)
    # Method 1 requires standardized inputs
    df_standardized = df_econ.copy()
    for col in df_standardized.columns:
        if col != 'prev_decision': # Keep categorical proxy as is (or standardize if preferred)
            df_standardized[col] = (df_standardized[col] - df_standardized[col].mean()) / df_standardized[col].std()
            
    return df_standardized


#### Unstructured data preprocess

In [3]:
def process_text_data(minutes_file, dictionary_file):
    print(">> Processing Text Data...")
    df = pd.read_csv(minutes_file)
    
    # A. Date Extraction & Alignment
    def extract_date(link):
        match = re.search(r'(\d{8})', link)
        if match: return pd.to_datetime(match.group(1), format='%Y%m%d')
        return None
    
    df['observation_date'] = df['Link'].apply(extract_date)
    df = df.dropna(subset=['observation_date']).sort_values('observation_date')
    
    # CRITICAL: Align Meeting Dates to Month Start (to match Economic Data)
    df['observation_date'] = df['observation_date'].dt.to_period('M').dt.to_timestamp()
    df = df.set_index('observation_date')

    # B. Text Cleaning (Method 1: Lowercase, No Punctuation, No Lemmatization)
    def clean_text(text):
        if not isinstance(text, str): return ""
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    df['Cleaned_Text'] = df['Text'].apply(clean_text)

    # C. TF-IDF Vectorization (500 Features)
    tfidf = TfidfVectorizer(stop_words='english', max_features=500, ngram_range=(1, 1))
    tfidf_matrix = tfidf.fit_transform(df['Cleaned_Text'])
    tfidf_cols = [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]
    df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_cols, index=df.index)

    # D. LM Sentiment Extraction
    print(">> Calculating Sentiment Scores...")
    lm_dict = pd.read_csv(dictionary_file)
    
    # Build Sentiment Sets (O(1) lookup)
    lm_pos = set(lm_dict[lm_dict['Positive'] > 0]['Word'].str.lower())
    lm_neg = set(lm_dict[lm_dict['Negative'] > 0]['Word'].str.lower())
    lm_unc = set(lm_dict[lm_dict['Uncertainty'] > 0]['Word'].str.lower())
    lm_lit = set(lm_dict[lm_dict['Litigious'] > 0]['Word'].str.lower())

    def get_sentiment(text):
        tokens = text.split()
        if not tokens: return pd.Series([0]*4, index=['LM_Pos','LM_Neg','LM_Unc','LM_Lit'])
        
        counts = {'pos':0, 'neg':0, 'unc':0, 'lit':0}
        for t in tokens:
            if t in lm_pos: counts['pos']+=1
            elif t in lm_neg: counts['neg']+=1
            elif t in lm_unc: counts['unc']+=1
            elif t in lm_lit: counts['lit']+=1
            
        total = len(tokens)
        return pd.Series({
            'LM_Positive': counts['pos']/total,
            'LM_Negative': counts['neg']/total,
            'LM_Uncertain': counts['unc']/total,
            'LM_Litigious': counts['lit']/total
        })

    df_sentiment = df['Cleaned_Text'].apply(get_sentiment)
    
    # Combine Text Features
    return pd.concat([df_tfidf, df_sentiment], axis=1)
